<a href="https://colab.research.google.com/github/aqilfiras/SMS-Classification/blob/main/SMS_Spam_CLS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 42.3 MB/s eta 0:00:00


In [2]:
import torch
import gensim
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

# Change device to 'cuda' if available

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pd.set_option('future.no_silent_downcasting', True)

# Get the data

In [4]:
data = pd.read_csv('spam.csv', encoding='latin-1')
data = data.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1)
data = data.rename(columns={'v1': 'label', 'v2': 'text'})
print(data['label'].value_counts())
data = data.replace({'ham': 0, 'spam': 1})
data.head()

FileNotFoundError: [Errno 2] No such file or directory: 'spam.csv'

# Set values for x and y

In [ ]:
y_raw,X_raw = data['label'].copy().tolist(), data['text'].copy().str.lower().tolist()

In [ ]:
X_raw

# Select and load model

In [ ]:
model = SentenceTransformer('sentence-transformers/all-roberta-large-v1', device = 'cuda')

# Convert the mails into embeddings

In [ ]:
sentence_embeddings = model.encode(X_raw, convert_to_tensor=True )

In [ ]:
sentence_embeddings

# Split into training and test set

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(sentence_embeddings, y_raw, test_size=0.2, random_state=42)


# Load and perform standard scaling on the training and test set

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.cpu())
X_test = scaler.transform(X_test.cpu())

# Convert into pytorch tensors

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32, device = device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32, device = device).unsqueeze(1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32, device = device)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32, device = device).unsqueeze(1)

In [ ]:
X_train_tensor.shape

# Build the logistic regression class

In [ ]:
class LogisticRegression(nn.Module):
  def __init__(self, input_dim):
    super(LogisticRegression, self).__init__()
    self.hidden = nn.Linear(input_dim, 16)
    self.relu = nn.ReLU()
    self.output = nn.Linear(16, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    x = self.hidden(x)
    x = self.relu(x)
    x = self.output(x)
    return x

# Load the model, criterion, and optimizer

In [ ]:
model = LogisticRegression(input_dim=1024).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training session

In [ ]:
epochs = 100
for epoch in range(epochs):
  optimizer.zero_grad()
  logits = model(X_train_tensor)
  loss = criterion(logits, y_train_tensor)
  loss.backward()
  optimizer.step()
  if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")


# Evaluate model on the test set

In [ ]:
model.eval()
with torch.no_grad():
    raw_logits = model(X_test_tensor)

    probabilities = torch.sigmoid(raw_logits)

    predictions = (probabilities >= 0.5).int()

In [ ]:
predictions

In [ ]:
disp = ConfusionMatrixDisplay.from_predictions(
    y_test_tensor.cpu(),
    predictions.cpu(),
    display_labels=['Ham', 'Spam'],
    cmap=plt.cm.Blues
)

plt.show()
print(classification_report(y_test_tensor.cpu(),predictions.cpu()))